---
title: "Privacy Auditing — Self-study notebook"
subtitle: "Applied Data Privacy"
format:
  html:
    page-layout: full
    toc: false
    toc-depth: 2
    code-tools: true
    code-fold: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display img, .cell-output-display .plotly-graph-div { max-width: 100%; height: auto; }
        </style>
execute:
  enabled: true
  warning: false
  message: false
jupyter: python3
---

This is the **self-study** companion to the lecture deck: full narrative + all code, meant to be
read and run. For the slide version (code hidden), open the [presentation deck](../../lecture-presentations/privacy-auditing/presentation.html).


In [ ]:
#| output: false
#| echo: false
# Environment setup: pick up local edits to libdpy/ without restarting the kernel,
# and install the public library when missing (e.g. on Google Colab).
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy


In [ ]:
#| echo: false
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from libdpy.visualization.privacy_plots import PrivacyPlot
from libdpy.visualization.roc_plots import (
    EmpiricalEpsilonFromDeltaVisualizer,
    TheoryROCVisualizer,
)
from libdpy.assignment_specific.privacy_auditing.visualizations import (
    NaiveSafeEpsilonHistogram,
)
from libdpy.assignment_specific.privacy_auditing.lecture_figures import (
    make_two_logistic_samplers,
    make_audit_confusion_matrix_figure,
    make_confidence_rectangle_figure,
    make_repeated_audit_points_figure,
    make_sampled_densities_figure,
    make_threshold_event_figure,
)
from libdpy.assignment_specific.privacy_auditing.utils import (
    audit_point,
    reference_point_two_logistic,
    selected_threshold_from_empirical_roc,
)

# Lecture contract — identical seeds/constants to the slide deck so every figure aligns.
delta_default = 1e-2
seed_empirical_roc = 4
n_empirical_roc = 500
alpha_total_default = 0.05

sampler_neg, sampler_pos = make_two_logistic_samplers(scale1=1.0, scale2=0.4)
rng_roc = np.random.default_rng(seed_empirical_roc)
samples_neg_roc = sampler_neg(n=n_empirical_roc, rng=rng_roc)
samples_pos_roc = sampler_pos(n=n_empirical_roc, rng=rng_roc)
tau_star, governing_point, eps_plug_roc = selected_threshold_from_empirical_roc(
    samples_neg_roc, samples_pos_roc, delta_default
)
audit_main = audit_point(
    samples_neg_roc, samples_pos_roc, tau_star, delta_default, alpha_total=alpha_total_default
)

# Privacy Auditing

In the previous lecture, we saw that differential privacy can be read as a constraint on the ROC curve of the best adversary distinguishing two neighboring datasets. This lecture asks what happens when we cannot compute that ROC exactly.

Auditing is the empirical version of the same geometry: run the mechanism on two neighboring datasets, estimate how well a distinguisher can separate the outputs, and convert the result into a **lower bound** on the privacy loss.

- Proofs give **upper bounds** on privacy loss.
- Audits give **lower bounds** on privacy loss.
- A failed audit does not prove privacy.
- A successful audit can falsify a claimed guarantee or expose a bug.

> The plots below are live, self-contained explorers (they run Python in your browser — the first one may take a moment to boot). Drag the sliders, press **Generate Samples** / **Resample**, and toggle **Compute ε**.

## Recap: DP as a bound on the ROC

Let $P = M(D)$ and $Q = M(D')$ be output distributions on neighboring datasets, and let $S$ be the event "the adversary guesses $D$" (positive).

$$\mathrm{TPR}=P(S),\qquad \mathrm{FPR}=Q(S).$$

For this lecture we keep one ROC direction fixed:

$$\mathrm{TPR} \le e^{\varepsilon}\,\mathrm{FPR}+\delta.$$

Drag $\varepsilon$ and $\delta$ to see the forbidden region the DP guarantee carves out of the ROC plane.

In [ ]:
#| label: fig-recall-dp-roc-line
#| fig-cap: "The DP guarantee defines a forbidden region in the ROC plane."
PrivacyPlot(
    sensitivity=1,
    std=1.5,
    distribution_types=[stats.norm, stats.laplace],
    epsilon=1.2,
    delta=0.05,
).embed()

## Exact inversion for known ROCs (recall, not auditing)

For a fixed $\delta$, any ROC point $(x,y)=(\mathrm{FPR},\mathrm{TPR})$ gives the one-sided requirement

$$\varepsilon \ge \log\left(\frac{y-\delta}{x}\right)$$

when $x>0$ and $y>\delta$. Known ROC + fixed $\delta$ $\rightarrow$ tightest DP line $\rightarrow$ exact $\varepsilon$.

This is still exact privacy-profile geometry, not empirical auditing.

### Laplace: exact ROC

For Laplace noise, the output distributions are shifts of the same known density. We can compute the ROC exactly and find the smallest $\varepsilon$ line for a chosen $\delta$. Check **Compute ε** to draw the tight line.

In [ ]:
#| label: fig-laplace-exact-eps
TheoryROCVisualizer(
    distribution="Laplace",
    scale=1.0,
    delta=1e-15,

    selectable_distribution=False,
).embed()

### Gaussian: approximate DP

Gaussian noise illustrates how changing $\delta$ changes the tight line. Drag the $\delta$ slider to compare.

In [ ]:
#| label: fig-gaussian-exact-eps
TheoryROCVisualizer(
    distribution="Gaussian",
    scale=1.0,
    delta=1e-2,

    selectable_distribution=False,
).embed()

### Logistic: adding another analytic distribution

The same ROC machinery is not tied to Laplace or Gaussian noise. If we can compute the PDF/CDF, we can add another distribution and repeat the exact construction.

$$f(x)=\frac{e^{-(x-\mu)/s}}{s(1+e^{-(x-\mu)/s})^2},\qquad F(x)=\frac{1}{1+e^{-(x-\mu)/s}}.$$

In [ ]:
#| label: fig-logistic-exact-eps
TheoryROCVisualizer(
    distribution="Logistic",
    scale=1.0,
    delta=1e-15,

    selectable_distribution=False,
).embed()

## From exact ROC to empirical ROC

Now define the noise procedurally:

$$Z = Z_1 + Z_2,\qquad Z_1 \sim \mathrm{Logistic}(0,s_1),\quad Z_2 \sim \mathrm{Logistic}(0,s_2),$$

with $Z_1,Z_2$ independent. The mechanism releases $M(D)=q(D)+Z$, giving

$$H_0: X=0+Z,\qquad H_1: X=1+Z.$$

Sampling is trivial. But we will not use a closed-form PDF, CDF, or ROC. From here on, the auditor treats the mechanism as a **black-box sampler** — it has samples, not formulas.

In [ ]:
#| label: fig-two-logistic-sampled-densities
#| fig-cap: "Samples from two shifted black-box output distributions."
make_sampled_densities_figure(samples_neg_roc, samples_pos_roc)
plt.show()

With samples we can draw an **empirical ROC**, but it is only an estimate: it moves from run to run. The explorer below runs the same empirical-ROC widget on the sum-of-two-logistics mechanism. Press **Generate Samples** to redraw; check **Compute ε** to highlight the governing point and the tight $(\varepsilon,\delta)$ line.

In [ ]:
#| label: fig-two-logistic-empirical-roc
EmpiricalEpsilonFromDeltaVisualizer(
    n_samples=500,
    distribution="Sum of logistic distributions",
    scale=1.0,
    delta=1e-2,
    random_seed=4,

    selectable_distribution=False,
).embed()

The empirical ROC is an *estimate* built from a finite sample. Watch it fill in as the 200 paired draws accumulate — early on it is jagged and unstable, then it settles toward the true curve:

In [ ]:
#| echo: false
#| output: asis
from libdpy.visualization.animation_embed import animation_markdown_image

print(
    animation_markdown_image(
        "blog-posts",
        "privacy-auditing",
        "empirical-roc",
        alt="Empirical ROC accumulating as samples arrive.",
    )
)

### Fixed-threshold audit unfolding (same sample as empirical ROC)

The threshold $\tau_\star$ is frozen from the empirical ROC above. The animation replays the **same** finite sample, with experiments arriving in a balanced random order shuffled by the same seed. The final audit point matches the governing point from the empirical ROC step. The animation plays once and stops on that point.

In [ ]:
#| echo: false
#| output: asis
from libdpy.visualization.animation_embed import animation_player_iframe

print(
    animation_player_iframe(
        "blog-posts",
        "privacy-auditing",
        "fixed-threshold-audit-same-sample",
        height=430,
    )
)

### Same threshold, fresh audit sample (audit seed 42)

The threshold is unchanged, but we redraw a new finite sample. Therefore the final counts and plug-in $\varepsilon$ can change.

In [ ]:
#| echo: false
#| output: asis
from libdpy.visualization.animation_embed import animation_player_iframe

print(
    animation_player_iframe(
        "blog-posts",
        "privacy-auditing",
        "fixed-threshold-audit-seed-42",
        height=430,
    )
)

### Threshold handoff for the rest of the lecture

The audit figures below need a **fixed** threshold $\tau_\star$ selected from the seeded empirical ROC. We read it off programmatically so every later cell uses exactly the same value.

In [ ]:
#| label: fig-two-logistic-selected-threshold
print(f"tau_star = {tau_star:.6f}")
print(f"governing point (FPR, TPR) = ({governing_point[0]:.3f}, {governing_point[1]:.3f})")
print(f"plug-in epsilon from empirical ROC = {eps_plug_roc:.4f}")

## From empirical ROC to one audit event

The empirical ROC step selected a threshold $\tau_\star$. We now freeze that threshold and audit the single event it defines.

$$S_{\tau_\star}=\{x:x>\tau_\star\},\qquad \mathrm{TPR}=P(X>\tau_\star\mid H_1),\qquad \mathrm{FPR}=P(X>\tau_\star\mid H_0).$$

In [ ]:
#| label: fig-threshold-event-two-logistic
make_threshold_event_figure(samples_neg_roc, samples_pos_roc, tau_star)
plt.show()

## Confusion matrix audit

Counting how many $H_0$ and $H_1$ samples land past $\tau_\star$ turns the event into a confusion matrix and a single plug-in ROC point.

$$\widehat{\varepsilon}_{\mathrm{plug}} = \log\left(\frac{\widehat{\mathrm{TPR}}-\delta}{\widehat{\mathrm{FPR}}}\right).$$

In [ ]:
#| label: fig-audit-confusion-matrix
make_audit_confusion_matrix_figure(audit_main)
plt.show()

## A single run is random

The mechanism did not change. The threshold did not change. Only the finite audit sample changed — so the plug-in $\varepsilon$ is a random variable. Each dot below is an independent fixed-threshold audit; the star is a large-simulation reference value not available to the auditor.

In [ ]:
#| label: fig-repeated-audit-points
rng_repeated = np.random.default_rng(456)
runs = []
for _ in range(40):
    neg = sampler_neg(n=500, rng=rng_repeated)
    pos = sampler_pos(n=500, rng=rng_repeated)
    runs.append(audit_point(neg, pos, tau_star, delta_default))
reference_point = reference_point_two_logistic(tau_star, delta_default, seed=999)
make_repeated_audit_points_figure(runs, reference=reference_point)
plt.show()

## Tail bounds and safe coordinates

For fixed $\tau_\star$, the counts are binomial. To lower-bound $\varepsilon$ we use a lower confidence bound on TPR and an upper confidence bound on FPR, splitting the total failure probability $\alpha$ across both coordinates.

$$\varepsilon_{\mathrm{audit}} = \log\left(\frac{\max\{\mathrm{TPR}_L-\delta,0\}}{\mathrm{FPR}_U}\right).$$

The code uses exact one-sided Clopper–Pearson intervals on each coordinate: move the empirical point **down** (lower TPR) and **right** (upper FPR) to the safe corner.

In [ ]:
#| label: fig-confidence-rectangle
print(
    f"safe FPR_U={audit_main.fpr_u:.4f}, TPR_L={audit_main.tpr_l:.4f}, "
    f"eps_audit={audit_main.eps_audit:.4f}"
)
make_confidence_rectangle_figure(audit_main)
plt.show()

## Naive versus safe $\varepsilon$

The plug-in estimate is random and overshoots often. The safe estimate is intentionally conservative.

The explorer below uses **Gaussian** outputs with a known analytic true $\varepsilon$. The title reports **overshoots** (runs where the estimate exceeds truth). Plug-in overshoots should be near 50%; safe overshoots should stay at or below the allowed failure rate $\alpha$ (often much lower because the bound is conservative). Drag $\alpha$ and the repetition count, and press **Resample**.

In [ ]:
#| label: fig-epsilon-histogram-naive-safe
NaiveSafeEpsilonHistogram(
    alpha_total=0.05,
    delta=1e-2,
    n_audit=500,
    n_repeats=200,
    seed=457,
).embed()

## Extra complications

1. **Threshold selection:** $\tau_\star$ was selected from a seeded empirical ROC so the audit figures align. With a new empirical ROC sample, the plug-in-optimal threshold may change. The simple confidence correction is valid for a threshold fixed **before** the audit sample; if the threshold is chosen using the same audit data, use held-out samples or uniform confidence bounds.
2. **Multiple neighboring pairs** and **multiple events/tests** all need statistical correction.
3. **Audits are lower bounds:** failing to find leakage does not prove privacy.
4. **Implementation bugs:** audits are useful because real implementations can violate their claimed theoretical guarantee.

## Summary

By the end of this lecture you should be able to state:

- DP can be read as an upper bound on the ROC curve of any membership distinguisher.
- If the ROC is known exactly, fixing $\delta$ determines the tight $\varepsilon$ line.
- Auditing begins when the ROC is not known and must be inferred by sampling.
- The seeded empirical ROC selects a threshold $\tau_\star$, which is then frozen for the audit.
- A fixed event gives $(\widehat{\mathrm{FPR}},\widehat{\mathrm{TPR}})$ and a plug-in lower bound on $\varepsilon$, but the plug-in value is random.
- A valid audit uses confidence bounds: lower TPR, upper FPR, with the total failure probability allocated across both bounds.
- The result is a high-confidence lower bound on privacy loss, not a proof of privacy.